In [1]:
import sagemaker

import boto3
from botocore.exceptions import ClientError
import ast

# secret shhhhh
def get_secret():

    secret_name = "gitlab/tcwyu-respository"
    region_name = "us-east-2"

    # Create a Secrets Manager client
    session = boto3.session.Session()
    client = session.client(
        service_name='secretsmanager',
        region_name=region_name
    )

    try:
        get_secret_value_response = client.get_secret_value(
            SecretId=secret_name
        )
    except ClientError as e:
        # For a list of exceptions thrown, see
        # https://docs.aws.amazon.com/secretsmanager/latest/apireference/API_GetSecretValue.html
        raise e

    # Decrypts secret using the associated KMS key.
    secret = get_secret_value_response['SecretString']

    # Your code goes here.
    return secret
    
secret = ast.literal_eval(get_secret())


In [2]:
# git config
git_config = {
    'repo': 'https://git.uwaterloo.ca/medical-ai-lab/bertha_experiments.git',
    'username': secret['username'],
    'password': secret['password']
}

hyperparameters = {
    'epochs': 50, 
    'batch_size': 64, 
    'lr': 2.5e-5,
}

estimator_args = {
    "base_job_name": "bertha-finetuning-mortality-uw-dataset",
    "role": "arn:aws:iam::776029861160:role/service-role/AmazonSageMaker-ExecutionRole-20230328T101238",
    "output_path": "s3://himss2023mimiciv/uw_skynet/output",
    "model_uri": "s3://himss2023mimiciv/uw_skynet/output/bertha-pretraining-uw-dataset-2023-04-06-03-48-44-106/output/model.tar.gz",
    "image_uri": "776029861160.dkr.ecr.us-east-2.amazonaws.com/smile-uw-containers:medair-huggingface-pytorch-training-test",
    #"checkpoint_s3_uri": "s3://himss2023mimiciv/uw_skynet/output/checkpoints",
    "instance_count": 1,
    "instance_type": "ml.p3.2xlarge",
    "entry_point": "BERTHA_AWS_Mortality.py",
    "git_config": git_config,
    "source_dir": '.',
    "hyperparameters": hyperparameters,
}

pytorch_estimator = sagemaker.estimator.Estimator(**estimator_args)

In [3]:
input_data = {
    "train": "s3://himss2023mimiciv/uw_skynet/data/mimic-iv_data/hf_dataset_finetune_mortality_prediction_train/train",
    "test": "s3://himss2023mimiciv/uw_skynet/data/mimic-iv_data/hf_dataset_finetune_mortality_prediction_train/test",
    "tokenizer": "s3://himss2023mimiciv/uw_skynet/data/tokenizer-mimic-iv-icd-final"
}

In [4]:
pytorch_estimator.fit(inputs=input_data)

Cloning into '/tmp/tmph24fe5k5'...
INFO:sagemaker:Creating training-job with name: smile-uw-testing-pretraining-2023-04-06-03-09-34-556


2023-04-06 03:09:35 Starting - Starting the training job...
2023-04-06 03:09:59 Starting - Preparing the instances for training.........
2023-04-06 03:11:28 Downloading - Downloading input data
2023-04-06 03:11:28 Training - Downloading the training image...........................
2023-04-06 03:16:00 Training - Training image download completed. Training in progress...bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
sed: can't read changehostname.c: No such file or directory
gcc: error: changehostname.c: No such file or directory
gcc: fatal error: no input files
compilation terminated.
gcc: error: changehostname.o: No such file or directory
ERROR: ld.so: object '/libchangehostname.so' from LD_PRELOAD cannot be preloaded (cannot open shared object file): ignored.
ERROR: ld.so: object '/libchangehostname.so' from LD_PRELOAD cannot be preloaded (cannot open shared object file): ignored.
2023-04-06 03:16:15,623 sagemaker-train